# Phase 3 — JEPA Downstream Probe Evaluation & Benchmark Comparison

This notebook evaluates both trained JEPA models through the **exact same Phase 2 harness**:
- **Phase 3a — JEPA-Temporal:** Multi-block temporal masking only.
- **Phase 3b — JEPA-SpatioTemporal:** Dual-axis temporal + field-group masking.

### Evaluation Suite (identical to Phase 2):
1. **Latent Caching:** Precomputes `[N, 256]` latents using `JEPAEncoderForEval` (mean-pooling over time).
2. **Task 1: Trend Prediction:** Linear `TrendHead` probe, Macro-F1 & Accuracy across all 5 stocks.
3. **Task 2: Contiguous Imputation:** 20-step contiguous masked inputs, masked test MSE & MAE across all 5 stocks.
4. **Task 3: Cross-Stock Transfer:** Source stock `sz000001` transferred to 4 target stocks with 20% fine-tuning budget.
5. **Consolidated Side-by-Side Benchmarks:** Compares JEPA-Temporal, JEPA-SpatioTemporal, and all 6 Phase 1 baselines.


In [ ]:
# 1. Colab Setup & Google Drive Mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

!pip install -q lightning pandas numpy torch scikit-learn
print('Environment ready.')


In [ ]:
# 2. Imports & Seed Setup
import os, sys, time
import numpy as np
import pandas as pd
import torch
from common import set_seed
from downstream_common import (
    ALL_STOCKS, LATENT_DIM, SEQ_LEN,
    train_trend_head_probe,
    compute_trend_labels_and_windows,
)
from jepa_common import (
    load_frozen_jepa_encoder,
    precompute_and_cache_jepa_latents,
    run_jepa_imputation_probe,
)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
os.makedirs('downstream_results', exist_ok=True)
JEPA_VARIANTS = ['3a', '3b']
VARIANT_NAMES = {'3a': 'JEPA-Temporal', '3b': 'JEPA-SpatioTemporal'}


In [ ]:
# 3. Step 0: Precompute & Cache Latents for All JEPA Models & Stocks
print('=' * 80)
print('  STEP 0: PRECOMPUTING & CACHING LATENTS')
print('=' * 80)

jepa_encoders = {}
for variant in JEPA_VARIANTS:
    model_name = VARIANT_NAMES[variant]
    print(f'\nLoading frozen encoder for {model_name} (variant {variant})...')
    encoder = load_frozen_jepa_encoder(variant=variant, device=device)
    jepa_encoders[variant] = encoder
    
    for stock in ALL_STOCKS:
        print(f'Caching latents for {model_name} / {stock}...')
        precompute_and_cache_jepa_latents(variant=variant, stock=stock, encoder=encoder, device=device)

print('\n✓ All JEPA latents successfully cached.')


In [ ]:
# 4. Step 1: Task 1 — Trend Prediction Probing (Macro-F1 & Accuracy)
print('=' * 80)
print('  TASK 1: TREND PREDICTION PROBING (Linear TrendHead Probe)')
print('=' * 80)

task1_rows = []
for variant in JEPA_VARIANTS:
    model_name = VARIANT_NAMES[variant]
    print(f'\nEvaluating Trend Prediction for {model_name}:')
    for stock in ALL_STOCKS:
        latents_dir = f'latents/JEPA_{variant}/{stock}'
        train_z = np.load(f'{latents_dir}/train_latents.npy')
        train_y = np.load(f'{latents_dir}/train_labels.npy')
        val_z   = np.load(f'{latents_dir}/val_latents.npy')
        val_y   = np.load(f'{latents_dir}/val_labels.npy')
        test_z  = np.load(f'{latents_dir}/test_latents.npy')
        test_y  = np.load(f'{latents_dir}/test_labels.npy')
        thetas  = np.load(f'{latents_dir}/thetas.npy')
        
        t0 = time.time()
        metrics = train_trend_head_probe(
            train_z, train_y, val_z, val_y, test_z, test_y,
            epochs=50, lr=1e-3, batch_size=256, device=device
        )
        elapsed = time.time() - t0
        print(f'  {model_name:<22} / {stock}: Macro-F1 = {metrics["macro_f1"]:.4f}, Acc = {metrics["accuracy"]:.4f} ({elapsed:.1f}s)')
        
        row = {
            'model': model_name,
            'stock': stock,
            'macro_f1': metrics['macro_f1'],
            'accuracy': metrics['accuracy'],
            'prec_down': metrics['precision_down'],
            'rec_down': metrics['recall_down'],
            'prec_stable': metrics['precision_stable'],
            'rec_stable': metrics['recall_stable'],
            'prec_up': metrics['precision_up'],
            'rec_up': metrics['recall_up'],
            'theta_down': float(thetas[0]),
            'theta_up': float(thetas[1]),
        }
        task1_rows.append(row)

df_task1_jepa = pd.DataFrame(task1_rows)
df_task1_jepa.to_csv('downstream_results/task1_jepa.csv', index=False)

# Append to master trend_prediction_results.csv if not already present
master_t1_path = 'downstream_results/trend_prediction_results.csv'
if os.path.exists(master_t1_path):
    df_master_t1 = pd.read_csv(master_t1_path)
    # Remove previous JEPA entries to avoid duplicates on re-runs
    df_master_t1 = df_master_t1[~df_master_t1['model'].isin(VARIANT_NAMES.values())]
    df_master_t1 = pd.concat([df_master_t1, df_task1_jepa], ignore_index=True)
    df_master_t1.to_csv(master_t1_path, index=False)
    print(f'\n✓ Updated {master_t1_path} with JEPA results.')


In [ ]:
# 5. Step 2: Task 2 — Contiguous Imputation Probing (Masked Test MSE & MAE)
print('=' * 80)
print('  TASK 2: CONTIGUOUS IMPUTATION (Masked MSE & MAE)')
print('=' * 80)

task2_rows = []
for variant in JEPA_VARIANTS:
    model_name = VARIANT_NAMES[variant]
    encoder = jepa_encoders[variant]
    print(f'\nEvaluating Contiguous Imputation for {model_name}:')
    for stock in ALL_STOCKS:
        t0 = time.time()
        metrics = run_jepa_imputation_probe(encoder=encoder, stock=stock, epochs=50, lr=1e-3, batch_size=256, device=device)
        elapsed = time.time() - t0
        print(f'  {model_name:<22} / {stock}: Masked MSE = {metrics["masked_test_mse"]:.4f}, MAE = {metrics["masked_test_mae"]:.4f} ({elapsed:.1f}s)')
        
        row = {
            'model': model_name,
            'stock': stock,
            'masked_test_mse': metrics['masked_test_mse'],
            'masked_test_mae': metrics['masked_test_mae'],
        }
        task2_rows.append(row)

df_task2_jepa = pd.DataFrame(task2_rows)
df_task2_jepa.to_csv('downstream_results/task2_jepa.csv', index=False)

# Append to master imputation_results.csv
master_t2_path = 'downstream_results/imputation_results.csv'
if os.path.exists(master_t2_path):
    df_master_t2 = pd.read_csv(master_t2_path)
    df_master_t2 = df_master_t2[~df_master_t2['model'].isin(VARIANT_NAMES.values())]
    df_master_t2 = pd.concat([df_master_t2, df_task2_jepa], ignore_index=True)
    df_master_t2.to_csv(master_t2_path, index=False)
    print(f'\n✓ Updated {master_t2_path} with JEPA results.')


In [ ]:
# 6. Step 3: Task 3 — Cross-Stock Transfer Probing (Source sz000001 -> 4 Target Stocks)
print('=' * 80)
print('  TASK 3: CROSS-STOCK TRANSFER (20% Target Train Budget)')
print('=' * 80)

TARGET_STOCKS = ['sz000002', 'sz000858', 'sz300147', 'sz002415']
task3_rows = []

for variant in JEPA_VARIANTS:
    model_name = VARIANT_NAMES[variant]
    source_encoder = jepa_encoders[variant]
    print(f'\nEvaluating Transfer for {model_name} (Source: sz000001):')
    
    for tgt_stock in TARGET_STOCKS:
        tgt_csv = f'data/{tgt_stock}-level10_processed.csv'
        df_tgt = pd.read_csv(tgt_csv)
        split_indices, split_labels, thetas = compute_trend_labels_and_windows(df_tgt, k=5, seq_len=100)
        tgt_features = df_tgt.iloc[:, 1:].values
        
        # Exactly 20% of target train split (first 20% temporally)
        n_train_full = len(split_indices['train'])
        n_train_20pct = int(n_train_full * 0.20)
        train_starts = split_indices['train'][:n_train_20pct]
        train_y = split_labels['train'][:n_train_20pct]
        val_starts = split_indices['val']
        val_y = split_labels['val']
        test_starts = split_indices['test']
        test_y = split_labels['test']
        
        def extract_z(starts):
            z_list = []
            for b in range(0, len(starts), 512):
                b_starts = starts[b : b + 512]
                b_win = np.stack([tgt_features[s : s + 100] for s in b_starts])
                b_t = torch.tensor(b_win, dtype=torch.float32, device=device)
                with torch.no_grad():
                    z_list.append(source_encoder(b_t).cpu().numpy())
            return np.concatenate(z_list, axis=0) if z_list else np.empty((0, LATENT_DIM))
            
        train_z = extract_z(train_starts)
        val_z   = extract_z(val_starts)
        test_z  = extract_z(test_starts)
        
        t0 = time.time()
        metrics = train_trend_head_probe(
            train_z, train_y, val_z, val_y, test_z, test_y,
            epochs=50, lr=1e-3, batch_size=256, device=device
        )
        elapsed = time.time() - t0
        print(f'  {model_name:<22} -> {tgt_stock}: Transfer Macro-F1 = {metrics["macro_f1"]:.4f}, Acc = {metrics["accuracy"]:.4f} ({elapsed:.1f}s)')
        
        row = {
            'model': model_name,
            'source_stock': 'sz000001',
            'target_stock': tgt_stock,
            'transfer_macro_f1': metrics['macro_f1'],
            'transfer_accuracy': metrics['accuracy'],
            'prec_down': metrics['precision_down'],
            'rec_down': metrics['recall_down'],
            'prec_stable': metrics['precision_stable'],
            'rec_stable': metrics['recall_stable'],
            'prec_up': metrics['precision_up'],
            'rec_up': metrics['recall_up'],
            'theta_down': float(thetas[0]),
            'theta_up': float(thetas[1]),
        }
        task3_rows.append(row)

df_task3_jepa = pd.DataFrame(task3_rows)
df_task3_jepa.to_csv('downstream_results/task3_jepa.csv', index=False)

# Append to master transfer_results.csv
master_t3_path = 'downstream_results/transfer_results.csv'
if os.path.exists(master_t3_path):
    df_master_t3 = pd.read_csv(master_t3_path)
    df_master_t3 = df_master_t3[~df_master_t3['model'].isin(VARIANT_NAMES.values())]
    df_master_t3 = pd.concat([df_master_t3, df_task3_jepa], ignore_index=True)
    df_master_t3.to_csv(master_t3_path, index=False)
    print(f'\n✓ Updated {master_t3_path} with JEPA results.')


In [ ]:
# 7. Step 4: Consolidated Side-by-Side Comparison Tables (Section 9 Benchmark)
print('=' * 100)
print('  FINAL BENCHMARK COMPARISON: 6 BASELINES vs. JEPA-TEMPORAL vs. JEPA-SPATIOTEMPORAL')
print('=' * 100)

# Task 1: Trend Prediction Macro-F1 Comparison
df_t1_all = pd.read_csv('downstream_results/trend_prediction_results.csv')
piv_t1 = df_t1_all.pivot(index='model', columns='stock', values='macro_f1')
piv_t1['Mean Macro-F1'] = piv_t1.mean(axis=1)
piv_t1 = piv_t1.sort_values(by='Mean Macro-F1', ascending=False)

print('\n--- TASK 1: TREND PREDICTION (Macro-F1 Ranking) ---')
display(piv_t1.round(4))

# Task 2: Contiguous Imputation MSE Comparison
df_t2_all = pd.read_csv('downstream_results/imputation_results.csv')
piv_t2_mse = df_t2_all.pivot(index='model', columns='stock', values='masked_test_mse')
piv_t2_mse['Mean Masked MSE'] = piv_t2_mse.mean(axis=1)
piv_t2_mse = piv_t2_mse.sort_values(by='Mean Masked MSE', ascending=True)

print('\n--- TASK 2: CONTIGUOUS IMPUTATION (Masked Test MSE Ranking, Lower is Better) ---')
display(piv_t2_mse.round(4))

# Task 3: Cross-Stock Transfer Macro-F1 Comparison
df_t3_all = pd.read_csv('downstream_results/transfer_results.csv')
piv_t3 = df_t3_all.pivot(index='model', columns='target_stock', values='transfer_macro_f1')
piv_t3['Headline Mean Macro-F1'] = piv_t3.mean(axis=1)
piv_t3 = piv_t3.sort_values(by='Headline Mean Macro-F1', ascending=False)

print('\n--- TASK 3: CROSS-STOCK TRANSFER (Headline Mean Macro-F1 Ranking) ---')
display(piv_t3.round(4))

# Save consolidated summary
with open('downstream_results/phase3_final_summary.txt', 'w') as f:
    f.write('=== PHASE 3 FINAL BENCHMARK SUMMARY ===\n\n')
    f.write('Task 1: Trend Prediction Macro-F1:\n')
    f.write(piv_t1.round(4).to_string() + '\n\n')
    f.write('Task 2: Contiguous Imputation Masked MSE:\n')
    f.write(piv_t2_mse.round(4).to_string() + '\n\n')
    f.write('Task 3: Cross-Stock Transfer Macro-F1:\n')
    f.write(piv_t3.round(4).to_string() + '\n')
print('\n✓ Successfully exported consolidated summary to downstream_results/phase3_final_summary.txt')
